In [0]:
# =============================================================================
# CELL 1: SETUP & IMPORTS
# Import all necessary libraries for complete ML pipeline
# =============================================================================

import os
import time
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    classification_report
)

# MLflow
import mlflow
import mlflow.sklearn

print("✅ All libraries imported successfully!\n")

# Create output directories in ML/AI folder
BASE_DIR = "/Workspace/Users/tayebielouai@gmail.com/ML/AI"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/results", exist_ok=True)
os.makedirs(f"{BASE_DIR}/metrics", exist_ok=True)

print(f"📁 Output directories created:")
print(f"   • Base: {BASE_DIR}")
print(f"   • Models: {BASE_DIR}/models")
print(f"   • Results: {BASE_DIR}/results")
print(f"   • Metrics: {BASE_DIR}/metrics")

print(f"\n🕐 Pipeline started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All libraries imported successfully!

📁 Output directories created:
   • Base: /Workspace/Users/tayebielouai@gmail.com/ML/AI
   • Models: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models
   • Results: /Workspace/Users/tayebielouai@gmail.com/ML/AI/results
   • Metrics: /Workspace/Users/tayebielouai@gmail.com/ML/AI/metrics

🕐 Pipeline started at: 2026-04-26 07:44:31


In [0]:
# =============================================================================
# CELL 2: DATA LOADING FROM DELTA TABLES
# Load and join all star schema tables
# =============================================================================

print("="*80)
print("LOADING DATA FROM DELTA TABLES")
print("="*80)

# Load all dimension and fact tables
print("\n📥 Loading tables from workspace.healthcare_analytics...\n")

fact_appointments = spark.table("workspace.healthcare_analytics.fact_appointments")
print(f"✓ fact_appointments: {fact_appointments.count():,} rows")

fact_treatments = spark.table("workspace.healthcare_analytics.fact_treatments")
print(f"✓ fact_treatments: {fact_treatments.count():,} rows")

fact_billing = spark.table("workspace.healthcare_analytics.fact_billing")
print(f"✓ fact_billing: {fact_billing.count():,} rows")

dim_patients = spark.table("workspace.healthcare_analytics.dim_patients")
print(f"✓ dim_patients: {dim_patients.count():,} rows")

dim_doctors = spark.table("workspace.healthcare_analytics.dim_doctors")
print(f"✓ dim_doctors: {dim_doctors.count():,} rows")

dim_hospitals = spark.table("workspace.healthcare_analytics.dim_hospitals")
print(f"✓ dim_hospitals: {dim_hospitals.count():,} rows")

dim_date = spark.table("workspace.healthcare_analytics.dim_date")
print(f"✓ dim_date: {dim_date.count():,} rows")

# Join all tables to create complete dataset
print("\n🔗 Joining tables to create complete dataset...\n")

# Start with fact_appointments (completed appointments only)
data = fact_appointments.filter(F.col("is_completed") == 1)

# Join with patients
data = data.join(dim_patients, "patient_id", "left")

# Join with doctors
data = data.join(dim_doctors, "doctor_id", "left")

# Join with hospitals
data = data.join(dim_hospitals, "hospital_id", "left")

# Join with treatments to get cost information
data = data.join(
    fact_treatments.select("appointment_id", "cost", "treatment_type"),
    "appointment_id",
    "left"
)

print(f"✅ Complete dataset created: {data.count():,} rows\n")

# Display schema
print("-"*80)
print("DATASET SCHEMA")
print("-"*80)
data.printSchema()

# Display sample
print("\n" + "-"*80)
print("SAMPLE DATA (5 rows)")
print("-"*80)
display(data.limit(5))

LOADING DATA FROM DELTA TABLES

📥 Loading tables from workspace.healthcare_analytics...

✓ fact_appointments: 150,000 rows
✓ fact_treatments: 127,315 rows
✓ fact_billing: 127,315 rows
✓ dim_patients: 50,000 rows
✓ dim_doctors: 9,717 rows
✓ dim_hospitals: 284 rows
✓ dim_date: 4,365 rows

🔗 Joining tables to create complete dataset...

✅ Complete dataset created: 127,315 rows

--------------------------------------------------------------------------------
DATASET SCHEMA
--------------------------------------------------------------------------------
root
 |-- appointment_id: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- date_id: long (nullable = true)
 |-- appointment_date: timestamp (nullable = true)
 |-- appointment_time: string (nullable = true)
 |-- appointment_hour: integer (nullable = true)
 |-- time_of_day: string (nullable = true)
 |-- reason_for_visit: string (nullable 

appointment_id,hospital_id,doctor_id,patient_id,date_id,appointment_date,appointment_time,appointment_hour,time_of_day,reason_for_visit,status,is_completed,is_cancelled,is_no_show,is_scheduled,is_emergency,first_name,last_name,gender,date_of_birth,age,age_group,contact_number,address,registration_date,years_registered,insurance_provider,first_name,last_name,specialization,experience_level,years_experience,reputation_score,reputation_category,public_medical_opinion_index,phone_number,email,hospital_id,hospital_branch,name,city,state,country,region,latitude,longitude,beds_total,beds_occupied,capacity_tier,occupancy_rate,occupancy_category,icu_available,ventilators_available,blood_bank_available,ambulance_capacity,emergency_level,trauma_center_level,current_emergency_load,surge_status,waiting_time_estimate_minutes,response_time_avg_minutes,specialties,hospital_website_url,road_accessibility,public_transport_access,average_travel_time_minutes,cost,treatment_type
APT-000000342,HOSP-ISR-00229,DOC-00008038,PAT-00033486,20260426,2026-04-26T00:00:00.000Z,14:45:00,14,Afternoon,Fever,completed,1,0,0,0,0,英,Berrocal,Female,1969-12-13,56,Middle Aged,051-940-2375,"605 Mccoy Forge Suite 040, New Joshua, HI 31111",2026-04-14,0.03285420944558522,Medicaid,Rocío,Carter,Dermatology,Mid-Level,6,3.06,Average,62.05,936-914-7061 x950,dr.rocío.carter@hadassahmedicalcente.com,HOSP-ISR-00229,Hadassah Medical Center,Hadassah Medical Center,Jerusalem,Jerusalem,Israel,Other,31.7974,35.1415,1000,872,Very Large,87.2,High,true,116,true,6,high,Level 2,medium,alert,201,14,"Ophthalmology, Neurology, Pediatrics, Dermatology, Cardiology",https://www.hadassah.org.il,8,4,18,181.56,consultation
APT-000000404,HOSP-PHI-00183,DOC-00006497,PAT-00016553,20260426,2026-04-26T00:00:00.000Z,11:00:00,11,Morning,Pregnancy Checkup,completed,1,0,0,0,0,Anaïs,मेहता,Male,1976-03-22,50,Middle Aged,(0131)4960556,"avenue Colette Labbé, 32959 Alexandre",2026-04-11,0.04106776180698152,NHS,Reginald,भट,Ophthalmology,Senior,17,3.62,Good,81.74,090-5580-5438,dr.reginald.भट@st.luke'smedicalcent.com,HOSP-PHI-00183,St. Luke's Medical Center,St. Luke's Medical Center,Manila,Metro Manila,Philippines,Asia,14.5547,121.0493,650,458,Large,70.46,Moderate,true,65,true,4,medium,Level 1,medium,normal,47,11,"Internal Medicine, Surgery, Psychiatry",https://www.stluke.com.ph,9,5,24,215.61,consultation
APT-000001760,HOSP-SAU-00236,DOC-00008334,PAT-00048834,20260426,2026-04-26T00:00:00.000Z,10:00:00,10,Morning,Cough and Cold,completed,1,0,0,0,0,Jasmine,Macdonald,Other,1948-03-04,78,Senior,+55 61 3298-3389,"Urbanización de Xavier Ureña 4, Alicante, 03175",2026-02-28,0.15605749486652978,Medicare,Alexia,Owen,Oncology,Mid-Level,11,3.34,Average,65.49,886-881-9843 x274,dr.alexia.owen@kingfahadmedicalcity.com,HOSP-SAU-00236,King Fahad Medical City,King Fahad Medical City,Riyadh,Riyadh,Saudi Arabia,Other,24.7581,46.6848,1295,996,Very Large,76.91,Moderate,true,180,true,8,critical,Level 2,low,normal,106,21,"Orthopedics, Internal Medicine, Obstetrics, Oncology, Ophthalmology, Gynecology",https://www.kfmc.med.sa,10,4,33,106.58,consultation
APT-000003450,HOSP-SPA-00027,DOC-00000936,PAT-00036303,20260426,2026-04-26T00:00:00.000Z,09:00:00,9,Morning,Lab Test Results,completed,1,0,0,0,0,Luiz Otávio,吴,Male,2014-03-19,12,Child,0442560936,"Cañada Geraldo Luque 35 Apt. 23 , Zamora, 26406",2026-04-26,0.0,Blue Cross,सीमा,최,Orthopedics,Mid-Level,7,2.54,Average,50.46,18287948839,dr.सीमा.최@hospitaluniversitari.com,HOSP-SPA-00027,Hospital Universitario 12 de Octubre,Hospital Universitario 12 de Octubre,Madrid,Madrid,Spain,Europe,40.3749,-3.7048,1365,1061,Very Large,77.73,Moderate,true,91,true,9,critical,Level 2,low,normal,110,18,"Orthopedics, Psychiatry, Ophthalmology, Neurology, Emergency Medicine, Obstetrics",https://www.comunidad.madrid/hospital/12octubre,6,5,29,719.14,diagnostic
APT-000004763,HOSP-CAN-00259,DOC-00008937,PAT-00005640,20260426,2026-04-26T00:00:00.000Z,10:30:00,10,Morning,Hearing Issues,completed,1,0,0,0,0,Shaun,Walsh,O

In [0]:
# =============================================================================
# CELL 3: DATA CLEANING & PREPROCESSING
# Handle missing values, duplicates, and data quality issues
# =============================================================================

import pandas as pd

print("="*80)
print("DATA CLEANING & PREPROCESSING")
print("="*80)

# Convert to Pandas for easier manipulation with sklearn
print("\n📊 Converting Spark DataFrame to Pandas...")
df = data.toPandas()
print(f"✓ Converted: {len(df):,} rows, {len(df.columns)} columns\n")

# Check data quality
print("-"*80)
print("DATA QUALITY SUMMARY - BEFORE CLEANING")
print("-"*80)
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nMissing values per column:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0].sort_values(ascending=False))
else:
    print("  No missing values found")

print(f"\nDuplicate rows: {df.duplicated().sum():,}")

# Remove duplicates
original_len = len(df)
df = df.drop_duplicates()
print(f"\n🧹 Removed {original_len - len(df):,} duplicate rows")

print("\n🔧 Handling missing values...")

# Fill missing values column by column
for col_name in df.columns:
    col_series = df[col_name]
    if pd.api.types.is_object_dtype(col_series) or pd.api.types.is_string_dtype(col_series):
        # Fill categorical with 'Unknown'
        df[col_name] = col_series.fillna('Unknown')
    elif pd.api.types.is_numeric_dtype(col_series):
        # Fill numeric with median
        df[col_name] = col_series.fillna(col_series.median())

print("   ✓ All missing values filled")

# Data quality after cleaning
print("\n" + "-"*80)
print("DATA QUALITY SUMMARY - AFTER CLEANING")
print("-"*80)
print(f"Total rows: {len(df):,}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n✅ Data cleaning completed!")

DATA CLEANING & PREPROCESSING

📊 Converting Spark DataFrame to Pandas...
✓ Converted: 127,315 rows, 68 columns

--------------------------------------------------------------------------------
DATA QUALITY SUMMARY - BEFORE CLEANING
--------------------------------------------------------------------------------
Total rows: 127,315
Total columns: 68

Missing values per column:
trauma_center_level    5739
dtype: int64

Duplicate rows: 0

🧹 Removed 0 duplicate rows

🔧 Handling missing values...
   ✓ All missing values filled

--------------------------------------------------------------------------------
DATA QUALITY SUMMARY - AFTER CLEANING
--------------------------------------------------------------------------------
Total rows: 127,315
Missing values: 0
Duplicate rows: 0

✅ Data cleaning completed!


In [0]:
# =============================================================================
# CELL 4: FEATURE ENGINEERING
# Create meaningful features for hospital recommendation
# =============================================================================

print("="*80)
print("FEATURE ENGINEERING")
print("="*80)

print(f"\nStarting with {len(df):,} rows and {len(df.columns)} columns\n")

# ===========================
# 1. Hospital Availability Score
# ===========================
print("1️⃣ Creating availability_score...")
if 'occupancy_rate' in df.columns:
    df['availability_score'] = 1 - df['occupancy_rate']
    print(f"   ✓ availability_score = 1 - occupancy_rate")
else:
    # Create synthetic availability score
    np.random.seed(42)
    df['availability_score'] = np.random.uniform(0.7, 0.95, len(df))
    print(f"   ✓ availability_score created (synthetic: 70-95%)")

# ===========================
# 2. Urgency Score
# ===========================
print("\n2️⃣ Encoding urgency level...")
if 'is_emergency' in df.columns:
    df['urgency_score'] = df['is_emergency']
else:
    # Map reason_for_visit to urgency
    emergency_keywords = ['emergency', 'urgent', 'critical', 'severe', 'acute']
    df['urgency_score'] = df['reason_for_visit'].apply(
        lambda x: 1 if any(keyword in str(x).lower() for keyword in emergency_keywords) else 0
    )
print(f"   ✓ urgency_score created (mean: {df['urgency_score'].mean():.3f})")

# ===========================
# 3. Distance Proxy
# ===========================
print("\n3️⃣ Creating distance proxy...")
if 'latitude' in df.columns and 'longitude' in df.columns:
    # Simple distance proxy using lat/lon variance
    df['distance_proxy'] = (
        (df['latitude'] - df['latitude'].mean())**2 + 
        (df['longitude'] - df['longitude'].mean())**2
    )**0.5
    # Normalize to 0-1
    df['distance_proxy'] = (df['distance_proxy'] - df['distance_proxy'].min()) / \
                           (df['distance_proxy'].max() - df['distance_proxy'].min())
else:
    # Random distance proxy
    np.random.seed(42)
    df['distance_proxy'] = np.random.uniform(0, 0.5, len(df))
print(f"   ✓ distance_proxy created (range: 0-0.5)")

# ===========================
# 4. Waiting Time Score
# ===========================
print("\n4️⃣ Creating waiting_time_score...")
if 'waiting_time_estimate_minutes' in df.columns:
    # Normalize: lower waiting time = higher score
    max_wait = df['waiting_time_estimate_minutes'].max()
    df['waiting_time_score'] = 1 - (df['waiting_time_estimate_minutes'] / max_wait)
else:
    np.random.seed(42)
    df['waiting_time_score'] = np.random.uniform(0.5, 1.0, len(df))
print(f"   ✓ waiting_time_score created (mean: {df['waiting_time_score'].mean():.3f})")

# ===========================
# 5. Doctor Quality Score
# ===========================
print("\n5️⃣ Creating doctor_quality_score...")
if 'reputation_score' in df.columns and 'years_experience' in df.columns:
    # Normalize both metrics
    rep_norm = (df['reputation_score'] - df['reputation_score'].min()) / \
               (df['reputation_score'].max() - df['reputation_score'].min())
    exp_norm = (df['years_experience'] - df['years_experience'].min()) / \
               (df['years_experience'].max() - df['years_experience'].min())
    df['doctor_quality_score'] = (rep_norm * 0.6) + (exp_norm * 0.4)
else:
    np.random.seed(42)
    df['doctor_quality_score'] = np.random.uniform(0.6, 0.95, len(df))
print(f"   ✓ doctor_quality_score created (mean: {df['doctor_quality_score'].mean():.3f})")

# ===========================
# 6. Hospital Capacity Score
# ===========================
print("\n6️⃣ Creating hospital_capacity_score...")
if 'beds_total' in df.columns:
    df['hospital_capacity_score'] = (df['beds_total'] - df['beds_total'].min()) / \
                                    (df['beds_total'].max() - df['beds_total'].min())
else:
    np.random.seed(42)
    df['hospital_capacity_score'] = np.random.uniform(0.4, 1.0, len(df))
print(f"   ✓ hospital_capacity_score created")

# ===========================
# 7. Encode Categorical Variables
# ===========================
print("\n7️⃣ Encoding categorical variables...")

# Select categorical columns to encode
categorical_features = []
if 'city' in df.columns:
    categorical_features.append('city')
if 'state' in df.columns:
    categorical_features.append('state')
if 'specialization' in df.columns:
    categorical_features.append('specialization')
if 'gender' in df.columns:
    categorical_features.append('gender')
if 'insurance_provider' in df.columns:
    categorical_features.append('insurance_provider')

# Label encode categorical features
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"   • Encoded {col} ({len(le.classes_)} unique values)")

print(f"\n✓ Encoded {len(categorical_features)} categorical features")

# ===========================
# Feature Summary
# ===========================
print("\n" + "-"*80)
print("FEATURE ENGINEERING SUMMARY")
print("-"*80)

engineered_features = [
    'availability_score', 'urgency_score', 'distance_proxy',
    'waiting_time_score', 'doctor_quality_score', 'hospital_capacity_score'
]

for feat in engineered_features:
    if feat in df.columns:
        print(f"{feat:30s}: mean={df[feat].mean():.3f}, std={df[feat].std():.3f}")

print(f"\n✅ Feature engineering completed!")
print(f"   Total features created: {len(engineered_features) + len(categorical_features)}")

FEATURE ENGINEERING

Starting with 127,315 rows and 68 columns

1️⃣ Creating availability_score...
   ✓ availability_score = 1 - occupancy_rate

2️⃣ Encoding urgency level...
   ✓ urgency_score created (mean: 0.071)

3️⃣ Creating distance proxy...
   ✓ distance_proxy created (range: 0-0.5)

4️⃣ Creating waiting_time_score...
   ✓ waiting_time_score created (mean: 0.596)

5️⃣ Creating doctor_quality_score...
   ✓ doctor_quality_score created (mean: 0.299)

6️⃣ Creating hospital_capacity_score...
   ✓ hospital_capacity_score created

7️⃣ Encoding categorical variables...
   • Encoded city (102 unique values)
   • Encoded state (97 unique values)
   • Encoded specialization (24 unique values)
   • Encoded gender (3 unique values)
   • Encoded insurance_provider (15 unique values)

✓ Encoded 5 categorical features

--------------------------------------------------------------------------------
FEATURE ENGINEERING SUMMARY
--------------------------------------------------------------------

In [0]:
# =============================================================================
# CELL 5: LABEL CREATION
# Create target variable for hospital recommendation
# =============================================================================

print("="*80)
print("LABEL CREATION")
print("="*80)

# Create composite hospital score
print("\n📊 Creating composite hospital_score...\n")

df['hospital_score'] = (
    df['availability_score'] * 0.25 +
    df['waiting_time_score'] * 0.20 +
    df['doctor_quality_score'] * 0.25 +
    df['hospital_capacity_score'] * 0.15 +
    (1 - df['distance_proxy']) * 0.15  # Closer is better
)

print("Hospital score components:")
print("  • Availability: 25%")
print("  • Waiting Time: 20%")
print("  • Doctor Quality: 25%")
print("  • Hospital Capacity: 15%")
print("  • Distance (proximity): 15%")

# Statistics
print(f"\nHospital Score Statistics:")
print(f"  • Mean: {df['hospital_score'].mean():.3f}")
print(f"  • Median: {df['hospital_score'].median():.3f}")
print(f"  • Std: {df['hospital_score'].std():.3f}")
print(f"  • Min: {df['hospital_score'].min():.3f}")
print(f"  • Max: {df['hospital_score'].max():.3f}")

# Create binary classification label (top 50% = best hospitals)
print("\n🎯 Creating binary label 'best_hospital'...\n")
threshold = df['hospital_score'].median()
df['best_hospital'] = (df['hospital_score'] >= threshold).astype(int)

print(f"Threshold (median): {threshold:.3f}")
print(f"\nLabel Distribution:")
label_counts = df['best_hospital'].value_counts().sort_index()
for label, count in label_counts.items():
    pct = (count / len(df)) * 100
    label_name = "Not Recommended" if label == 0 else "Recommended"
    print(f"  • {label_name} (class {label}): {count:,} ({pct:.1f}%)")

# Check for class imbalance
class_ratio = label_counts.max() / label_counts.min()
print(f"\nClass balance ratio: {class_ratio:.2f}:1")
if class_ratio > 1.5:
    print("  ⚠️  Classes are imbalanced (consider using balanced weights)")
else:
    print("  ✅ Classes are well balanced")

print("\n✅ Label creation completed!")

LABEL CREATION

📊 Creating composite hospital_score...

Hospital score components:
  • Availability: 25%
  • Waiting Time: 20%
  • Doctor Quality: 25%
  • Hospital Capacity: 15%
  • Distance (proximity): 15%

Hospital Score Statistics:
  • Mean: -18.842
  • Median: -18.607
  • Std: 1.920
  • Min: -22.535
  • Max: -15.545

🎯 Creating binary label 'best_hospital'...

Threshold (median): -18.607

Label Distribution:
  • Not Recommended (class 0): 63,656 (50.0%)
  • Recommended (class 1): 63,659 (50.0%)

Class balance ratio: 1.00:1
  ✅ Classes are well balanced

✅ Label creation completed!


In [0]:
# =============================================================================
# CELL 6: FEATURE SELECTION & TRAIN/TEST SPLIT
# Prepare final feature set and split data
# =============================================================================

print("="*80)
print("FEATURE SELECTION & TRAIN/TEST SPLIT")
print("="*80)

# Define feature columns
feature_columns = [
    'availability_score',
    'urgency_score',
    'distance_proxy',
    'waiting_time_score',
    'doctor_quality_score',
    'hospital_capacity_score'
]

# Add encoded categorical features
for col in ['city', 'state', 'specialization', 'gender', 'insurance_provider']:
    if f'{col}_encoded' in df.columns:
        feature_columns.append(f'{col}_encoded')

print(f"\n📋 Selected Features ({len(feature_columns)} total):\n")
for i, feat in enumerate(feature_columns, 1):
    print(f"  {i:2d}. {feat}")

# Prepare X and y
X = df[feature_columns].copy()
y = df['best_hospital'].copy()

print(f"\n📊 Dataset shape:")
print(f"  • Features (X): {X.shape}")
print(f"  • Labels (y): {y.shape}")
print(f"  • Total samples: {len(X):,}")

# Handle any remaining NaN values
if X.isnull().sum().sum() > 0:
    print(f"\n⚠️  Found {X.isnull().sum().sum()} NaN values in features, filling with 0...")
    X = X.fillna(0)

# Train/Test Split
print("\n✂️  Splitting data into train and test sets...\n")

TRAIN_SIZE = 0.8
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y  # Maintain class distribution
)

print(f"Split configuration:")
print(f"  • Train size: {TRAIN_SIZE*100:.0f}%")
print(f"  • Test size: {TEST_SIZE*100:.0f}%")
print(f"  • Random state: {RANDOM_STATE}")
print(f"  • Stratified: Yes")

print(f"\n📊 Split results:")
print(f"  • X_train: {X_train.shape} ({len(X_train):,} samples)")
print(f"  • X_test: {X_test.shape} ({len(X_test):,} samples)")
print(f"  • y_train: {y_train.shape}")
print(f"  • y_test: {y_test.shape}")

# Check class distribution in splits
print(f"\n🎯 Class distribution:")
print(f"\nTraining set:")
train_dist = y_train.value_counts(normalize=True).sort_index()
for label, pct in train_dist.items():
    label_name = "Not Recommended" if label == 0 else "Recommended"
    print(f"  • {label_name}: {pct*100:.1f}%")

print(f"\nTest set:")
test_dist = y_test.value_counts(normalize=True).sort_index()
for label, pct in test_dist.items():
    label_name = "Not Recommended" if label == 0 else "Recommended"
    print(f"  • {label_name}: {pct*100:.1f}%")

# Feature Scaling
print("\n⚖️  Applying StandardScaler to features...\n")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Features scaled (mean=0, std=1)")
print(f"  • Train mean: {X_train_scaled.mean():.6f}")
print(f"  • Train std: {X_train_scaled.std():.6f}")

print("\n✅ Data preparation completed!")
print(f"\nReady for model training with:")
print(f"  • {len(feature_columns)} features")
print(f"  • {len(X_train):,} training samples")
print(f"  • {len(X_test):,} test samples")

FEATURE SELECTION & TRAIN/TEST SPLIT

📋 Selected Features (11 total):

   1. availability_score
   2. urgency_score
   3. distance_proxy
   4. waiting_time_score
   5. doctor_quality_score
   6. hospital_capacity_score
   7. city_encoded
   8. state_encoded
   9. specialization_encoded
  10. gender_encoded
  11. insurance_provider_encoded

📊 Dataset shape:
  • Features (X): (127315, 11)
  • Labels (y): (127315,)
  • Total samples: 127,315

✂️  Splitting data into train and test sets...

Split configuration:
  • Train size: 80%
  • Test size: 20%
  • Random state: 42
  • Stratified: Yes

📊 Split results:
  • X_train: (101852, 11) (101,852 samples)
  • X_test: (25463, 11) (25,463 samples)
  • y_train: (101852,)
  • y_test: (25463,)

🎯 Class distribution:

Training set:
  • Not Recommended: 50.0%
  • Recommended: 50.0%

Test set:
  • Not Recommended: 50.0%
  • Recommended: 50.0%

⚖️  Applying StandardScaler to features...

✓ Features scaled (mean=0, std=1)
  • Train mean: 0.000000
  • Tra

In [0]:
# =============================================================================
# CELL 7: MODEL 1 - RANDOM FOREST CLASSIFIER
# Train Random Forest model with optimized hyperparameters
# =============================================================================

print("="*80)
print("MODEL 1: RANDOM FOREST CLASSIFIER")
print("="*80)

print("\n🌲 Configuring Random Forest Classifier...\n")

# Model configuration
rf_params = {
    'n_estimators': 100,
    'max_depth': 15,
    'min_samples_split': 10,
    'min_samples_leaf': 4,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
    'class_weight': 'balanced'  # Handle any class imbalance
}

print("Model Hyperparameters:")
for param, value in rf_params.items():
    print(f"  • {param}: {value}")

# Initialize model
rf_classifier = RandomForestClassifier(**rf_params)

print("\n🔄 Training Random Forest model...\n")
start_time = time.time()

# Train model
rf_classifier.fit(X_train_scaled, y_train)

training_time = time.time() - start_time

print(f"✅ Random Forest training completed!")
print(f"   Training time: {training_time:.2f} seconds\n")

# Make predictions
print("🔮 Making predictions on test set...\n")

y_pred_rf = rf_classifier.predict(X_test_scaled)
y_pred_proba_rf = rf_classifier.predict_proba(X_test_scaled)

print(f"✓ Predictions completed")
print(f"  • Predicted classes shape: {y_pred_rf.shape}")
print(f"  • Predicted probabilities shape: {y_pred_proba_rf.shape}")

# Feature importance
print("\n🎯 Top 10 Most Important Features:\n")
feature_importance_rf = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_classifier.feature_importances_
}).sort_values('importance', ascending=False)

for idx, row in feature_importance_rf.head(10).iterrows():
    print(f"  {row['feature']:30s}: {row['importance']:.4f}")

print("\n✅ Random Forest model ready for evaluation!")

MODEL 1: RANDOM FOREST CLASSIFIER

🌲 Configuring Random Forest Classifier...

Model Hyperparameters:
  • n_estimators: 100
  • max_depth: 15
  • min_samples_split: 10
  • min_samples_leaf: 4
  • max_features: sqrt
  • random_state: 42
  • n_jobs: -1
  • class_weight: balanced

🔄 Training Random Forest model...

✅ Random Forest training completed!
   Training time: 2.34 seconds

🔮 Making predictions on test set...

✓ Predictions completed
  • Predicted classes shape: (25463,)
  • Predicted probabilities shape: (25463, 2)

🎯 Top 10 Most Important Features:

  availability_score            : 0.7098
  waiting_time_score            : 0.1918
  distance_proxy                : 0.0270
  hospital_capacity_score       : 0.0226
  state_encoded                 : 0.0210
  city_encoded                  : 0.0169
  doctor_quality_score          : 0.0094
  specialization_encoded        : 0.0012
  insurance_provider_encoded    : 0.0002
  gender_encoded                : 0.0001

✅ Random Forest model ready

In [0]:
# =============================================================================
# CELL 8: MODEL 2 - LOGISTIC REGRESSION
# Train Logistic Regression model
# =============================================================================

print("="*80)
print("MODEL 2: LOGISTIC REGRESSION")
print("="*80)

print("\n📊 Configuring Logistic Regression...\n")

# Model configuration
lr_params = {
    'max_iter': 1000,
    'C': 1.0,  # Regularization strength
    'penalty': 'l2',
    'solver': 'lbfgs',
    'random_state': 42,
    'class_weight': 'balanced',
    'n_jobs': -1
}

print("Model Hyperparameters:")
for param, value in lr_params.items():
    print(f"  • {param}: {value}")

# Initialize model
lr_classifier = LogisticRegression(**lr_params)

print("\n🔄 Training Logistic Regression model...\n")
start_time = time.time()

# Train model
lr_classifier.fit(X_train_scaled, y_train)

training_time = time.time() - start_time

print(f"✅ Logistic Regression training completed!")
print(f"   Training time: {training_time:.2f} seconds\n")

# Make predictions
print("🔮 Making predictions on test set...\n")

y_pred_lr = lr_classifier.predict(X_test_scaled)
y_pred_proba_lr = lr_classifier.predict_proba(X_test_scaled)

print(f"✓ Predictions completed")
print(f"  • Predicted classes shape: {y_pred_lr.shape}")
print(f"  • Predicted probabilities shape: {y_pred_proba_lr.shape}")

# Coefficient analysis
print("\n🎯 Top 10 Most Influential Features (by coefficient magnitude):\n")
feature_coefficients_lr = pd.DataFrame({
    'feature': feature_columns,
    'coefficient': lr_classifier.coef_[0],
    'abs_coefficient': np.abs(lr_classifier.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

for idx, row in feature_coefficients_lr.head(10).iterrows():
    direction = "positive" if row['coefficient'] > 0 else "negative"
    print(f"  {row['feature']:30s}: {row['coefficient']:+.4f} ({direction})")

print("\n✅ Logistic Regression model ready for evaluation!")

MODEL 2: LOGISTIC REGRESSION

📊 Configuring Logistic Regression...

Model Hyperparameters:
  • max_iter: 1000
  • C: 1.0
  • penalty: l2
  • solver: lbfgs
  • random_state: 42
  • class_weight: balanced
  • n_jobs: -1

🔄 Training Logistic Regression model...

✅ Logistic Regression training completed!
   Training time: 1.56 seconds

🔮 Making predictions on test set...

✓ Predictions completed
  • Predicted classes shape: (25463,)
  • Predicted probabilities shape: (25463, 2)

🎯 Top 10 Most Influential Features (by coefficient magnitude):

  availability_score            : +41.4006 (positive)
  distance_proxy                : -1.1736 (negative)
  waiting_time_score            : +1.0142 (positive)
  hospital_capacity_score       : +0.8407 (positive)
  doctor_quality_score          : +0.6175 (positive)
  city_encoded                  : -0.1702 (negative)
  urgency_score                 : +0.1181 (positive)
  insurance_provider_encoded    : -0.0290 (negative)
  specialization_encoded       

In [0]:
# ================================================================================================================
# CELL 9: MODEL EVALUATION & COMPARISON
# Comprehensive evaluation metrics for both models
# =============================================================================

print("="*80)
print("MODEL EVALUATION & COMPARISON")
print("="*80)

def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
    """Calculate comprehensive evaluation metrics"""
    
    # Basic metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary')
    recall = recall_score(y_true, y_pred, average='binary')
    f1 = f1_score(y_true, y_pred, average='binary')
    
    # ROC-AUC
    roc_auc = roc_auc_score(y_true, y_pred_proba[:, 1])
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Specificity
    specificity = tn / (tn + fp)
    
    metrics = {
        'model': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'specificity': specificity,
        'true_negatives': tn,
        'false_positives': fp,
        'false_negatives': fn,
        'true_positives': tp
    }
    
    return metrics, cm

# Evaluate Random Forest
print("\n🌲 Evaluating Random Forest Classifier...\n")
rf_metrics, rf_cm = evaluate_model(y_test, y_pred_rf, y_pred_proba_rf, "Random Forest")

print("Random Forest Results:")
print(f"  • Accuracy:    {rf_metrics['accuracy']:.4f} ({rf_metrics['accuracy']*100:.2f}%)")
print(f"  • Precision:   {rf_metrics['precision']:.4f}")
print(f"  • Recall:      {rf_metrics['recall']:.4f}")
print(f"  • F1-Score:    {rf_metrics['f1_score']:.4f}")
print(f"  • ROC-AUC:     {rf_metrics['roc_auc']:.4f}")
print(f"  • Specificity: {rf_metrics['specificity']:.4f}")

# Evaluate Logistic Regression
print("\n📊 Evaluating Logistic Regression...\n")
lr_metrics, lr_cm = evaluate_model(y_test, y_pred_lr, y_pred_proba_lr, "Logistic Regression")

print("Logistic Regression Results:")
print(f"  • Accuracy:    {lr_metrics['accuracy']:.4f} ({lr_metrics['accuracy']*100:.2f}%)")
print(f"  • Precision:   {lr_metrics['precision']:.4f}")
print(f"  • Recall:      {lr_metrics['recall']:.4f}")
print(f"  • F1-Score:    {lr_metrics['f1_score']:.4f}")
print(f"  • ROC-AUC:     {lr_metrics['roc_auc']:.4f}")
print(f"  • Specificity: {lr_metrics['specificity']:.4f}")

# Create comparison DataFrame
print("\n" + "="*80)
print("SIDE-BY-SIDE COMPARISON")
print("="*80 + "\n")

comparison_df = pd.DataFrame([rf_metrics, lr_metrics])
comparison_df = comparison_df[[
    'model', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'specificity'
]]

display(comparison_df)

# Determine best model
print("\n🏆 BEST MODEL SELECTION\n")

if rf_metrics['f1_score'] > lr_metrics['f1_score']:
    best_model = "Random Forest"
    best_f1 = rf_metrics['f1_score']
    best_model_obj = rf_classifier
    best_pred = y_pred_rf
    best_proba = y_pred_proba_rf
else:
    best_model = "Logistic Regression"
    best_f1 = lr_metrics['f1_score']
    best_model_obj = lr_classifier
    best_pred = y_pred_lr
    best_proba = y_pred_proba_lr

print(f"Best Model: {best_model}")
print(f"F1-Score: {best_f1:.4f}")

# Confusion matrices
print("\n" + "="*80)
print("CONFUSION MATRICES")
print("="*80)

print("\nRandom Forest:")
print("              Predicted")
print("              Neg   Pos")
print(f"Actual Neg   {rf_cm[0][0]:5d} {rf_cm[0][1]:5d}")
print(f"       Pos   {rf_cm[1][0]:5d} {rf_cm[1][1]:5d}")

print("\nLogistic Regression:")
print("              Predicted")
print("              Neg   Pos")
print(f"Actual Neg   {lr_cm[0][0]:5d} {lr_cm[0][1]:5d}")
print(f"       Pos   {lr_cm[1][0]:5d} {lr_cm[1][1]:5d}")

print("\n✅ Model evaluation completed!")

MODEL EVALUATION & COMPARISON

🌲 Evaluating Random Forest Classifier...

Random Forest Results:
  • Accuracy:    1.0000 (100.00%)
  • Precision:   1.0000
  • Recall:      1.0000
  • F1-Score:    1.0000
  • ROC-AUC:     1.0000
  • Specificity: 1.0000

📊 Evaluating Logistic Regression...

Logistic Regression Results:
  • Accuracy:    0.9968 (99.68%)
  • Precision:   0.9942
  • Recall:      0.9995
  • F1-Score:    0.9968
  • ROC-AUC:     1.0000
  • Specificity: 0.9942

SIDE-BY-SIDE COMPARISON



model,accuracy,precision,recall,f1_score,roc_auc,specificity
Random Forest,1.0,1.0,1.0,1.0,1.0,1.0
Logistic Regression,0.9968189137179437,0.994218298304555,0.9994502042098649,0.9968273863146763,0.9999893701746423,0.9941874165422984



🏆 BEST MODEL SELECTION

Best Model: Random Forest
F1-Score: 1.0000

CONFUSION MATRICES

Random Forest:
              Predicted
              Neg   Pos
Actual Neg   12731     0
       Pos       0 12732

Logistic Regression:
              Predicted
              Neg   Pos
Actual Neg   12657    74
       Pos       7 12725

✅ Model evaluation completed!


In [0]:
# =============================================================================
# CELL 9A: CROSS-VALIDATION SETUP
# Configure cross-validation strategy for model optimization
# =============================================================================

print("="*80)
print("CROSS-VALIDATION SETUP")
print("="*80)

# Import cross-validation tools
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV

print("\n📊 Setting up Stratified K-Fold Cross-Validation...\n")

# Configure cross-validation
CV_FOLDS = 5
CV_RANDOM_STATE = 42

# Create stratified k-fold cross-validator
cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=CV_RANDOM_STATE
)

print("Cross-Validation Configuration:")
print(f"  • Strategy: Stratified K-Fold")
print(f"  • Number of folds: {CV_FOLDS}")
print(f"  • Shuffle: Yes")
print(f"  • Random state: {CV_RANDOM_STATE}")

print("\n📖 What is Cross-Validation?")
print("-" * 80)
print("Cross-validation splits the training data into K folds, trains on K-1 folds,")
print("and validates on the remaining fold. This process repeats K times, rotating")
print("which fold is used for validation. This provides:")
print("  1. More reliable performance estimates")
print("  2. Better detection of overfitting")
print("  3. Optimal hyperparameter selection")

print("\n✅ Cross-validation setup completed!")
print("\nNext: Hyperparameter tuning for Random Forest and Logistic Regression...")

CROSS-VALIDATION SETUP

📊 Setting up Stratified K-Fold Cross-Validation...

Cross-Validation Configuration:
  • Strategy: Stratified K-Fold
  • Number of folds: 5
  • Shuffle: Yes
  • Random state: 42

📖 What is Cross-Validation?
--------------------------------------------------------------------------------
Cross-validation splits the training data into K folds, trains on K-1 folds,
and validates on the remaining fold. This process repeats K times, rotating
which fold is used for validation. This provides:
  1. More reliable performance estimates
  2. Better detection of overfitting
  3. Optimal hyperparameter selection

✅ Cross-validation setup completed!

Next: Hyperparameter tuning for Random Forest and Logistic Regression...


In [0]:
# =============================================================================
# CELL 9B: HYPERPARAMETER TUNING - RANDOM FOREST
# Use GridSearchCV to find optimal Random Forest hyperparameters
# =============================================================================

import time
from sklearn.ensemble import RandomForestClassifier

print("="*80)
print("HYPERPARAMETER TUNING - RANDOM FOREST")
print("="*80)

print("\n🔍 Setting up hyperparameter grid for Random Forest...\n")

# Define hyperparameter grid
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [5, 10, 15],
    'min_samples_leaf': [2, 4, 6],
    'max_features': ['sqrt'],
    'class_weight': ['balanced'],
    'random_state': [42]
}

print("Hyperparameter Search Space:")
for param, values in rf_param_grid.items():
    print(f"  • {param}: {values}")

total_combinations = (
    len(rf_param_grid['n_estimators']) *
    len(rf_param_grid['max_depth']) *
    len(rf_param_grid['min_samples_split']) *
    len(rf_param_grid['min_samples_leaf'])
)
print(f"\nTotal combinations to test: {total_combinations}")
print(f"With 3-fold CV: {total_combinations * 3} model fits\n")

# Create GridSearchCV
print("🔄 Starting GridSearchCV (this may take several minutes)...\n")

rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(n_jobs=-1),
    param_grid=rf_param_grid,
    cv=3,  # 3-fold CV for faster computation
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Fit grid search
start_time = time.time()
rf_grid_search.fit(X_train_scaled, y_train)
tuning_time = time.time() - start_time

print(f"\n✅ Grid search completed in {tuning_time:.2f} seconds\n")

# Best parameters
print("="*80)
print("BEST HYPERPARAMETERS FOUND")
print("="*80)
for param, value in rf_grid_search.best_params_.items():
    print(f"  • {param}: {value}")

print(f"\nBest CV F1-Score: {rf_grid_search.best_score_:.4f}")

# Create optimized model
optimized_rf_classifier = rf_grid_search.best_estimator_

# Evaluate on test set
y_pred_rf_opt = optimized_rf_classifier.predict(X_test_scaled)
y_pred_proba_rf_opt = optimized_rf_classifier.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
rf_optimized_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_rf_opt),
    'precision': precision_score(y_test, y_pred_rf_opt),
    'recall': recall_score(y_test, y_pred_rf_opt),
    'f1_score': f1_score(y_test, y_pred_rf_opt),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_rf_opt)
}

# Display results
print("\n" + "="*80)
print("OPTIMIZED MODEL PERFORMANCE (Test Set)")
print("="*80)
for metric, value in rf_optimized_metrics.items():
    print(f"{metric.replace('_', ' ').title():<15s}: {value:.4f}")

print("\n✅ Random Forest hyperparameter tuning completed!")

HYPERPARAMETER TUNING - RANDOM FOREST

🔍 Setting up hyperparameter grid for Random Forest...

Hyperparameter Search Space:
  • n_estimators: [50, 100, 200]
  • max_depth: [10, 15, 20]
  • min_samples_split: [5, 10, 15]
  • min_samples_leaf: [2, 4, 6]
  • max_features: ['sqrt']
  • class_weight: ['balanced']
  • random_state: [42]

Total combinations to test: 81
With 3-fold CV: 243 model fits

🔄 Starting GridSearchCV (this may take several minutes)...

Fitting 3 folds for each of 81 candidates, totalling 243 fits

✅ Grid search completed in 515.67 seconds

BEST HYPERPARAMETERS FOUND
  • class_weight: balanced
  • max_depth: 10
  • max_features: sqrt
  • min_samples_leaf: 2
  • min_samples_split: 10
  • n_estimators: 50
  • random_state: 42

Best CV F1-Score: 1.0000

OPTIMIZED MODEL PERFORMANCE (Test Set)
Accuracy       : 1.0000
Precision      : 1.0000
Recall         : 1.0000
F1 Score       : 1.0000
Roc Auc        : 1.0000

✅ Random Forest hyperparameter tuning completed!


In [0]:
# =============================================================================
# CELL 9C: HYPERPARAMETER TUNING - LOGISTIC REGRESSION
# Use GridSearchCV to find optimal Logistic Regression hyperparameters
# =============================================================================

from sklearn.linear_model import LogisticRegression

print("="*80)
print("HYPERPARAMETER TUNING - LOGISTIC REGRESSION")
print("="*80)

print("\n🔍 Setting up hyperparameter grid for Logistic Regression...\n")

# Define hyperparameter grid
lr_param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [1000],
    'class_weight': ['balanced'],
    'random_state': [42]
}

print("Hyperparameter Search Space:")
for param, values in lr_param_grid.items():
    print(f"  • {param}: {values}")

total_combinations = (
    len(lr_param_grid['C']) *
    len(lr_param_grid['penalty']) *
    len(lr_param_grid['solver'])
)
print(f"\nTotal combinations to test: {total_combinations}")
print(f"With 3-fold CV: {total_combinations * 3} model fits\n")

# Create GridSearchCV
print("🔄 Starting GridSearchCV...\n")

lr_grid_search = GridSearchCV(
    estimator=LogisticRegression(n_jobs=-1),
    param_grid=lr_param_grid,
    cv=3,  # 3-fold CV for faster computation
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Fit grid search
start_time = time.time()
lr_grid_search.fit(X_train_scaled, y_train)
tuning_time = time.time() - start_time

print(f"\n✅ Grid search completed in {tuning_time:.2f} seconds\n")

# Best parameters
print("="*80)
print("BEST HYPERPARAMETERS FOUND")
print("="*80)
for param, value in lr_grid_search.best_params_.items():
    print(f"  • {param}: {value}")

print(f"\nBest CV F1-Score: {lr_grid_search.best_score_:.4f}")

# Create optimized model
optimized_lr_classifier = lr_grid_search.best_estimator_

# Evaluate on test set
y_pred_lr_opt = optimized_lr_classifier.predict(X_test_scaled)
y_pred_proba_lr_opt = optimized_lr_classifier.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
lr_optimized_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_lr_opt),
    'precision': precision_score(y_test, y_pred_lr_opt),
    'recall': recall_score(y_test, y_pred_lr_opt),
    'f1_score': f1_score(y_test, y_pred_lr_opt),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_lr_opt)
}

# Display results
print("\n" + "="*80)
print("OPTIMIZED MODEL PERFORMANCE (Test Set)")
print("="*80)
for metric, value in lr_optimized_metrics.items():
    print(f"{metric.replace('_', ' ').title():<15s}: {value:.4f}")

print("\n✅ Logistic Regression hyperparameter tuning completed!")

HYPERPARAMETER TUNING - LOGISTIC REGRESSION

🔍 Setting up hyperparameter grid for Logistic Regression...

Hyperparameter Search Space:
  • C: [0.01, 0.1, 1.0, 10.0]
  • penalty: ['l1', 'l2']
  • solver: ['liblinear', 'saga']
  • max_iter: [1000]
  • class_weight: ['balanced']
  • random_state: [42]

Total combinations to test: 16
With 3-fold CV: 48 model fits

🔄 Starting GridSearchCV...

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is 


✅ Grid search completed in 105.85 seconds

BEST HYPERPARAMETERS FOUND
  • C: 10.0
  • class_weight: balanced
  • max_iter: 1000
  • penalty: l1
  • random_state: 42
  • solver: liblinear

Best CV F1-Score: 0.9997

OPTIMIZED MODEL PERFORMANCE (Test Set)
Accuracy       : 0.9995
Precision      : 0.9994
Recall         : 0.9997
F1 Score       : 0.9995
Roc Auc        : 1.0000

✅ Logistic Regression hyperparameter tuning completed!


In [0]:
# =============================================================================
# CELL 9D: CROSS-VALIDATION RESULTS COMPARISON
# Compare all models using 5-fold cross-validation
# =============================================================================

print("="*80)
print("CROSS-VALIDATION RESULTS COMPARISON")
print("="*80)

print("\n🔄 Running 5-fold cross-validation on all models...\n")
print("This provides a robust estimate of model performance.\n")

# Dictionary to store all models
models_to_compare = {
    'Random Forest (Baseline)': rf_classifier,
    'Random Forest (Optimized)': optimized_rf_classifier,
    'Logistic Regression (Baseline)': lr_classifier,
    'Logistic Regression (Optimized)': optimized_lr_classifier
}

# Store CV results
cv_results = []

for model_name, model in models_to_compare.items():
    print(f"Evaluating {model_name}...")
    
    # Perform 5-fold cross-validation
    cv_scores = cross_val_score(
        model, X_train_scaled, y_train,
        cv=cv_strategy,
        scoring='f1',
        n_jobs=-1
    )
    
    # Calculate statistics
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Get test set performance
    if 'Random Forest (Baseline)' == model_name:
        test_acc = rf_metrics['accuracy']
        test_f1 = rf_metrics['f1_score']
    elif 'Random Forest (Optimized)' == model_name:
        test_acc = rf_optimized_metrics['accuracy']
        test_f1 = rf_optimized_metrics['f1_score']
    elif 'Logistic Regression (Baseline)' == model_name:
        test_acc = lr_metrics['accuracy']
        test_f1 = lr_metrics['f1_score']
    else:
        test_acc = lr_optimized_metrics['accuracy']
        test_f1 = lr_optimized_metrics['f1_score']
    
    cv_results.append({
        'Model': model_name,
        'CV_Mean_F1': cv_mean,
        'CV_Std_F1': cv_std,
        'Test_Accuracy': test_acc,
        'Test_F1': test_f1
    })
    
    print(f"  ✓ CV F1-Score: {cv_mean:.4f} (±{cv_std:.4f})")

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS TABLE")
print("="*80 + "\n")

# Create results DataFrame
cv_results_df = pd.DataFrame(cv_results)

# Calculate improvement from baseline
rf_baseline_cv = cv_results_df[cv_results_df['Model'] == 'Random Forest (Baseline)']['CV_Mean_F1'].values[0]
lr_baseline_cv = cv_results_df[cv_results_df['Model'] == 'Logistic Regression (Baseline)']['CV_Mean_F1'].values[0]

cv_results_df['Improvement_vs_Baseline'] = cv_results_df.apply(
    lambda row: ((row['CV_Mean_F1'] - rf_baseline_cv) / rf_baseline_cv * 100) if 'Random Forest' in row['Model']
    else ((row['CV_Mean_F1'] - lr_baseline_cv) / lr_baseline_cv * 100) if 'Logistic' in row['Model']
    else 0,
    axis=1
)

# Display results
display(cv_results_df)

# Find best model
best_model_idx = cv_results_df['CV_Mean_F1'].idxmax()
best_model_name = cv_results_df.loc[best_model_idx, 'Model']
best_cv_f1 = cv_results_df.loc[best_model_idx, 'CV_Mean_F1']
best_test_f1 = cv_results_df.loc[best_model_idx, 'Test_F1']

print("\n" + "="*80)
print("🏆 BEST OVERALL MODEL")
print("="*80)
print(f"\nModel: {best_model_name}")
print(f"CV F1-Score: {best_cv_f1:.4f}")
print(f"Test F1-Score: {best_test_f1:.4f}")
print(f"\nThis model shows the best generalization performance based on")
print(f"cross-validation and will be used for production deployment.")

# Update best model variables
if 'Random Forest (Optimized)' == best_model_name:
    best_model = "Random Forest (Optimized)"
    best_model_obj = optimized_rf_classifier
    best_f1 = rf_optimized_metrics['f1_score']
    best_metrics = rf_optimized_metrics
elif 'Logistic Regression (Optimized)' == best_model_name:
    best_model = "Logistic Regression (Optimized)"
    best_model_obj = optimized_lr_classifier
    best_f1 = lr_optimized_metrics['f1_score']
    best_metrics = lr_optimized_metrics
elif 'Random Forest (Baseline)' == best_model_name:
    best_model = "Random Forest (Baseline)"
    best_model_obj = rf_classifier
    best_f1 = rf_metrics['f1_score']
    best_metrics = rf_metrics
else:
    best_model = "Logistic Regression (Baseline)"
    best_model_obj = lr_classifier
    best_f1 = lr_metrics['f1_score']
    best_metrics = lr_metrics

print("\n✅ Cross-validation comparison completed!")

CROSS-VALIDATION RESULTS COMPARISON

🔄 Running 5-fold cross-validation on all models...

This provides a robust estimate of model performance.

Evaluating Random Forest (Baseline)...
  ✓ CV F1-Score: 1.0000 (±0.0000)
Evaluating Random Forest (Optimized)...
  ✓ CV F1-Score: 1.0000 (±0.0000)
Evaluating Logistic Regression (Baseline)...
  ✓ CV F1-Score: 0.9967 (±0.0004)
Evaluating Logistic Regression (Optimized)...


/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is 

  ✓ CV F1-Score: 0.9997 (±0.0001)

COMPREHENSIVE RESULTS TABLE



Model,CV_Mean_F1,CV_Std_F1,Test_Accuracy,Test_F1,Improvement_vs_Baseline
Random Forest (Baseline),0.9999901821216435,1.9635756712998998E-5,1.0,1.0,0.0
Random Forest (Optimized),0.9999803632793324,3.927344133529686E-5,1.0,1.0,-9.818938712250391E-4
Logistic Regression (Baseline),0.9967016042088439,4.2587488163107134E-4,0.9968189137179437,0.9968273863146763,0.0
Logistic Regression (Optimized),0.9997054848470832,9.315632735304322E-5,0.9995287279582139,0.9995288204806031,0.30138214141068953



🏆 BEST OVERALL MODEL

Model: Random Forest (Baseline)
CV F1-Score: 1.0000
Test F1-Score: 1.0000

This model shows the best generalization performance based on
cross-validation and will be used for production deployment.

✅ Cross-validation comparison completed!


In [0]:
# =============================================================================
# CELL 9E: SAVE OPTIMIZED MODELS & TUNING RESULTS
# Save hyperparameter-tuned models and all optimization results
# =============================================================================

print("="*80)
print("SAVING OPTIMIZED MODELS & TUNING RESULTS")
print("="*80)

print(f"\n💾 Base directory: {BASE_DIR}\n")

# ====================================
# 1. Save Optimized Random Forest
# ====================================
print("1️⃣ Saving optimized Random Forest model...")
rf_optimized_path = f"{BASE_DIR}/models/random_forest_optimized.pkl"
with open(rf_optimized_path, 'wb') as f:
    pickle.dump(optimized_rf_classifier, f)
print(f"   ✓ Saved: {rf_optimized_path}")

# ====================================
# 2. Save Optimized Logistic Regression
# ====================================
print("\n2️⃣ Saving optimized Logistic Regression model...")
lr_optimized_path = f"{BASE_DIR}/models/logistic_regression_optimized.pkl"
with open(lr_optimized_path, 'wb') as f:
    pickle.dump(optimized_lr_classifier, f)
print(f"   ✓ Saved: {lr_optimized_path}")

# ====================================
# 3. Save Hyperparameter Tuning Results
# ====================================
print("\n3️⃣ Saving hyperparameter tuning results...")

# Random Forest tuning results
rf_tuning_results = pd.DataFrame({
    'Model': ['Random Forest'],
    'Best_n_estimators': [rf_grid_search.best_params_['n_estimators']],
    'Best_max_depth': [rf_grid_search.best_params_['max_depth']],
    'Best_min_samples_split': [rf_grid_search.best_params_['min_samples_split']],
    'Best_min_samples_leaf': [rf_grid_search.best_params_['min_samples_leaf']],
    'Best_CV_F1_Score': [rf_grid_search.best_score_],
    'Baseline_F1_Score': [rf_metrics['f1_score']],
    'Optimized_Test_F1_Score': [rf_optimized_metrics['f1_score']],
    'F1_Improvement': [rf_optimized_metrics['f1_score'] - rf_metrics['f1_score']]
})

# Logistic Regression tuning results
lr_tuning_results = pd.DataFrame({
    'Model': ['Logistic Regression'],
    'Best_C': [lr_grid_search.best_params_['C']],
    'Best_penalty': [lr_grid_search.best_params_['penalty']],
    'Best_solver': [lr_grid_search.best_params_['solver']],
    'Best_CV_F1_Score': [lr_grid_search.best_score_],
    'Baseline_F1_Score': [lr_metrics['f1_score']],
    'Optimized_Test_F1_Score': [lr_optimized_metrics['f1_score']],
    'F1_Improvement': [lr_optimized_metrics['f1_score'] - lr_metrics['f1_score']]
})

tuning_results_path = f"{BASE_DIR}/results/hyperparameter_tuning_results.csv"
rf_tuning_results.to_csv(tuning_results_path, index=False)
lr_tuning_results.to_csv(tuning_results_path, mode='a', header=False, index=False)
print(f"   ✓ Saved: {tuning_results_path}")

# ====================================
# 4. Save Cross-Validation Scores
# ====================================
print("\n4️⃣ Saving cross-validation scores...")
cv_scores_path = f"{BASE_DIR}/results/cross_validation_scores.csv"
cv_results_df.to_csv(cv_scores_path, index=False)
print(f"   ✓ Saved: {cv_scores_path}")

# ====================================
# 5. Save Best Model Info
# ====================================
print("\n5️⃣ Saving best model information...")
best_model_info = {
    'best_model_name': best_model,
    'best_cv_f1_score': float(best_cv_f1),
    'best_test_f1_score': float(best_test_f1),
    'best_test_accuracy': float(best_metrics['accuracy']),
    'best_test_precision': float(best_metrics['precision']),
    'best_test_recall': float(best_metrics['recall']),
    'best_test_roc_auc': float(best_metrics['roc_auc']),
    'optimization_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

import json
best_model_info_path = f"{BASE_DIR}/models/best_model_info.json"
with open(best_model_info_path, 'w') as f:
    json.dump(best_model_info, f, indent=2)
print(f"   ✓ Saved: {best_model_info_path}")

# ====================================
# Summary
# ====================================
print("\n" + "="*80)
print("✅ ALL OPTIMIZATION RESULTS SAVED!")
print("="*80)

print(f"\n📦 Optimized Models:")
print(f"  • {rf_optimized_path}")
print(f"  • {lr_optimized_path}")

print(f"\n📊 Tuning Results:")
print(f"  • {tuning_results_path}")
print(f"  • {cv_scores_path}")
print(f"  • {best_model_info_path}")

print(f"\n🏆 Best Model: {best_model}")
print(f"   Test F1-Score: {best_test_f1:.4f}")
print(f"   CV F1-Score: {best_cv_f1:.4f}")

print("\n👉 All files saved to: /Workspace/Users/tayebielouai@gmail.com/ml_models/hospital_recommendation/")
print("\n✅ Hyperparameter optimization completed! Proceeding to MLflow tracking...")

SAVING OPTIMIZED MODELS & TUNING RESULTS

💾 Base directory: /Workspace/Users/tayebielouai@gmail.com/ML/AI

1️⃣ Saving optimized Random Forest model...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/random_forest_optimized.pkl

2️⃣ Saving optimized Logistic Regression model...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/logistic_regression_optimized.pkl

3️⃣ Saving hyperparameter tuning results...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/results/hyperparameter_tuning_results.csv

4️⃣ Saving cross-validation scores...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/results/cross_validation_scores.csv

5️⃣ Saving best model information...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/best_model_info.json

✅ ALL OPTIMIZATION RESULTS SAVED!

📦 Optimized Models:
  • /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/random_forest_optimized.pkl
  • /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/logist

In [0]:
# =============================================================================
# CELL 10: MLFLOW EXPERIMENT TRACKING
# Log all models (baseline + optimized), parameters, and metrics to MLflow
# =============================================================================

print("="*80)
print("MLFLOW EXPERIMENT TRACKING")
print("="*80)

# Set experiment name
experiment_name = "/Users/tayebielouai@gmail.com/Hospital_Recommendation_ML"

try:
    mlflow.set_experiment(experiment_name)
    print(f"\n✓ MLflow experiment set: {experiment_name}\n")
    
    # ====================================
    # 1. Log Baseline Random Forest
    # ====================================
    print("🌲 Logging Baseline Random Forest to MLflow...")
    with mlflow.start_run(run_name="Random_Forest_Baseline"):
        # Log parameters
        mlflow.log_param("model_type", "RandomForestClassifier")
        mlflow.log_param("n_estimators", 100)
        mlflow.log_param("max_depth", 15)
        mlflow.log_param("optimization", "baseline")
        
        # Log metrics
        mlflow.log_metric("accuracy", rf_metrics['accuracy'])
        mlflow.log_metric("precision", rf_metrics['precision'])
        mlflow.log_metric("recall", rf_metrics['recall'])
        mlflow.log_metric("f1_score", rf_metrics['f1_score'])
        mlflow.log_metric("roc_auc", rf_metrics['roc_auc'])
        
        # Log model
        mlflow.sklearn.log_model(rf_classifier, "random_forest_baseline")
        print("   ✓ Baseline Random Forest logged successfully")
    
    # ====================================
    # 2. Log Optimized Random Forest
    # ====================================
    print("\n🌲 Logging Optimized Random Forest to MLflow...")
    with mlflow.start_run(run_name="Random_Forest_Optimized"):
        # Log parameters
        mlflow.log_param("model_type", "RandomForestClassifier")
        best_rf_params = rf_grid_search.best_params_
        for param_name, param_value in best_rf_params.items():
            mlflow.log_param(param_name, param_value)
        mlflow.log_param("optimization", "grid_search_cv")
        mlflow.log_param("cv_folds", CV_FOLDS)
        
        # Log metrics
        mlflow.log_metric("accuracy", rf_optimized_metrics['accuracy'])
        mlflow.log_metric("precision", rf_optimized_metrics['precision'])
        mlflow.log_metric("recall", rf_optimized_metrics['recall'])
        mlflow.log_metric("f1_score", rf_optimized_metrics['f1_score'])
        mlflow.log_metric("roc_auc", rf_optimized_metrics['roc_auc'])
        mlflow.log_metric("cv_best_score", rf_grid_search.best_score_)
        
        # Log model
        mlflow.sklearn.log_model(optimized_rf_classifier, "random_forest_optimized")
        print("   ✓ Optimized Random Forest logged successfully")
    
    # ====================================
    # 3. Log Baseline Logistic Regression
    # ====================================
    print("\n📊 Logging Baseline Logistic Regression to MLflow...")
    with mlflow.start_run(run_name="Logistic_Regression_Baseline"):
        # Log parameters
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("C", 1.0)
        mlflow.log_param("penalty", "l2")
        mlflow.log_param("optimization", "baseline")
        
        # Log metrics
        mlflow.log_metric("accuracy", lr_metrics['accuracy'])
        mlflow.log_metric("precision", lr_metrics['precision'])
        mlflow.log_metric("recall", lr_metrics['recall'])
        mlflow.log_metric("f1_score", lr_metrics['f1_score'])
        mlflow.log_metric("roc_auc", lr_metrics['roc_auc'])
        
        # Log model
        mlflow.sklearn.log_model(lr_classifier, "logistic_regression_baseline")
        print("   ✓ Baseline Logistic Regression logged successfully")
    
    # ====================================
    # 4. Log Optimized Logistic Regression
    # ====================================
    print("\n📊 Logging Optimized Logistic Regression to MLflow...")
    with mlflow.start_run(run_name="Logistic_Regression_Optimized"):
        # Log parameters
        mlflow.log_param("model_type", "LogisticRegression")
        best_lr_params = lr_grid_search.best_params_
        for param_name, param_value in best_lr_params.items():
            mlflow.log_param(param_name, param_value)
        mlflow.log_param("optimization", "grid_search_cv")
        mlflow.log_param("cv_folds", CV_FOLDS)
        
        # Log metrics
        mlflow.log_metric("accuracy", lr_optimized_metrics['accuracy'])
        mlflow.log_metric("precision", lr_optimized_metrics['precision'])
        mlflow.log_metric("recall", lr_optimized_metrics['recall'])
        mlflow.log_metric("f1_score", lr_optimized_metrics['f1_score'])
        mlflow.log_metric("roc_auc", lr_optimized_metrics['roc_auc'])
        mlflow.log_metric("cv_best_score", lr_grid_search.best_score_)
        
        # Log model
        mlflow.sklearn.log_model(optimized_lr_classifier, "logistic_regression_optimized")
        print("   ✓ Optimized Logistic Regression logged successfully")
    
    print("\n" + "="*80)
    print("✅ ALL 4 MODELS LOGGED TO MLFLOW SUCCESSFULLY")
    print("="*80)
    print(f"\n🔗 View experiments at: Experiments > {experiment_name}")
    
except Exception as e:
    print(f"\n⚠️ MLflow logging failed: {str(e)}")
    print("Continuing without MLflow logging...")

print("\n✓ MLflow tracking completed")

In [0]:
# =============================================================================
# CELL 11: SAVE MODELS & RESULTS TO WORKSPACE
# Save all models (baseline + optimized), metrics, and predictions as files
# =============================================================================

print("="*80)
print("SAVING MODELS & RESULTS TO WORKSPACE")
print("="*80)

print(f"\n💾 Base directory: {BASE_DIR}\n")

# ====================================
# 1. Save Baseline Random Forest
# ====================================
print("1️⃣ Saving Baseline Random Forest model...")
rf_baseline_path = f"{BASE_DIR}/models/random_forest_baseline.pkl"
with open(rf_baseline_path, 'wb') as f:
    pickle.dump(rf_classifier, f)
print(f"   ✓ Saved: {rf_baseline_path}")

# ====================================
# 2. Save Optimized Random Forest
# ====================================
print("\n2️⃣ Saving Optimized Random Forest model...")
rf_optimized_path = f"{BASE_DIR}/models/random_forest_optimized.pkl"
with open(rf_optimized_path, 'wb') as f:
    pickle.dump(optimized_rf_classifier, f)
print(f"   ✓ Saved: {rf_optimized_path}")

# ====================================
# 3. Save Baseline Logistic Regression
# ====================================
print("\n3️⃣ Saving Baseline Logistic Regression model...")
lr_baseline_path = f"{BASE_DIR}/models/logistic_regression_baseline.pkl"
with open(lr_baseline_path, 'wb') as f:
    pickle.dump(lr_classifier, f)
print(f"   ✓ Saved: {lr_baseline_path}")

# ====================================
# 4. Save Optimized Logistic Regression
# ====================================
print("\n4️⃣ Saving Optimized Logistic Regression model...")
lr_optimized_path = f"{BASE_DIR}/models/logistic_regression_optimized.pkl"
with open(lr_optimized_path, 'wb') as f:
    pickle.dump(optimized_lr_classifier, f)
print(f"   ✓ Saved: {lr_optimized_path}")

# ====================================
# 5. Save Best Model (based on CV)
# ====================================
print(f"\n5️⃣ Saving best model: {best_model}...")
best_model_path = f"{BASE_DIR}/models/best_model.pkl"
with open(best_model_path, 'wb') as f:
    pickle.dump(best_model_obj, f)
print(f"   ✓ Saved: {best_model_path}")

# ====================================
# 6. Save Scaler
# ====================================
print("\n6️⃣ Saving feature scaler...")
scaler_path = f"{BASE_DIR}/models/scaler.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"   ✓ Saved: {scaler_path}")

# ====================================
# 7. Save All Metrics
# ====================================
print("\n7️⃣ Saving model metrics comparison...")
metrics_comparison = pd.DataFrame({
    'Model': ['RF Baseline', 'RF Optimized', 'LR Baseline', 'LR Optimized'],
    'Accuracy': [
        rf_metrics['accuracy'],
        rf_optimized_metrics['accuracy'],
        lr_metrics['accuracy'],
        lr_optimized_metrics['accuracy']
    ],
    'Precision': [
        rf_metrics['precision'],
        rf_optimized_metrics['precision'],
        lr_metrics['precision'],
        lr_optimized_metrics['precision']
    ],
    'Recall': [
        rf_metrics['recall'],
        rf_optimized_metrics['recall'],
        lr_metrics['recall'],
        lr_optimized_metrics['recall']
    ],
    'F1-Score': [
        rf_metrics['f1_score'],
        rf_optimized_metrics['f1_score'],
        lr_metrics['f1_score'],
        lr_optimized_metrics['f1_score']
    ],
    'ROC-AUC': [
        rf_metrics['roc_auc'],
        rf_optimized_metrics['roc_auc'],
        lr_metrics['roc_auc'],
        lr_optimized_metrics['roc_auc']
    ]
})

metrics_path = f"{BASE_DIR}/results/all_models_metrics.csv"
metrics_comparison.to_csv(metrics_path, index=False)
print(f"   ✓ Saved: {metrics_path}")

# ====================================
# 8. Save Predictions
# ====================================
print("\n8️⃣ Saving test predictions from best model...")
y_pred_best = best_model_obj.predict(X_test_scaled)
predictions_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred_best,
    'prediction_proba': best_model_obj.predict_proba(X_test_scaled)[:, 1]
})

predictions_path = f"{BASE_DIR}/results/test_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)
print(f"   ✓ Saved: {predictions_path}")

# ====================================
# 9. Save Best Model Info
# ====================================
print("\n9️⃣ Saving best model metadata...")
best_model_info = {
    'model_name': best_model,
    'f1_score': float(best_f1),
    'cv_f1_score': float(best_cv_f1),
    'metrics': {
        'accuracy': float(best_metrics['accuracy']),
        'precision': float(best_metrics['precision']),
        'recall': float(best_metrics['recall']),
        'f1_score': float(best_metrics['f1_score']),
        'roc_auc': float(best_metrics['roc_auc'])
    },
    'file_path': f"{BASE_DIR}/models/" + best_model.lower().replace(' ', '_').replace('(', '').replace(')', '') + ".pkl"
}

import json
info_path = f"{BASE_DIR}/results/best_model_info.json"
with open(info_path, 'w') as f:
    json.dump(best_model_info, f, indent=2)
print(f"   ✓ Saved: {info_path}")

print("\n" + "="*80)
print("✅ ALL MODELS AND RESULTS SAVED SUCCESSFULLY")
print("="*80)
print(f"\n📂 Saved files in: {BASE_DIR}")
print("\n   Models:")
print("     • random_forest_baseline.pkl")
print("     • random_forest_optimized.pkl")
print("     • logistic_regression_baseline.pkl")
print("     • logistic_regression_optimized.pkl")
print("     • best_model.pkl")
print("     • scaler.pkl")
print("\n   Results:")
print("     • all_models_metrics.csv")
print("     • test_predictions.csv")
print("     • best_model_info.json")

SAVING MODELS & RESULTS TO WORKSPACE

💾 Base directory: /Workspace/Users/tayebielouai@gmail.com/ML/AI

1️⃣ Saving Baseline Random Forest model...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/random_forest_baseline.pkl

2️⃣ Saving Optimized Random Forest model...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/random_forest_optimized.pkl

3️⃣ Saving Baseline Logistic Regression model...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/logistic_regression_baseline.pkl

4️⃣ Saving Optimized Logistic Regression model...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/logistic_regression_optimized.pkl

5️⃣ Saving best model: Random Forest (Baseline)...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/best_model.pkl

6️⃣ Saving feature scaler...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/scaler.pkl

7️⃣ Saving model metrics comparison...
   ✓ Saved: /Workspace/Users/tayebielouai@gmail.

In [0]:
# =============================================================================
# CELL 12: MODEL COMPARISON & RECOMMENDATIONS
# Detailed analysis including baseline and optimized models
# =============================================================================

print("="*80)
print("MODEL COMPARISON & RECOMMENDATIONS")
print("="*80)

# ====================================
# 1. Comprehensive Performance Table
# ====================================
print("\n📈 COMPREHENSIVE MODEL COMPARISON (All 4 Models)\n")
print("="*80)
print(f"{'Model':<30s} {'Accuracy':>10s} {'Precision':>10s} {'Recall':>10s} {'F1-Score':>10s} {'ROC-AUC':>10s}")
print("="*80)

all_models_comparison = [
    ('RF Baseline', rf_metrics),
    ('RF Optimized', rf_optimized_metrics),
    ('LR Baseline', lr_metrics),
    ('LR Optimized', lr_optimized_metrics)
]

for model_name, metrics in all_models_comparison:
    print(f"{model_name:<30s} {metrics['accuracy']:>10.4f} {metrics['precision']:>10.4f} {metrics['recall']:>10.4f} {metrics['f1_score']:>10.4f} {metrics['roc_auc']:>10.4f}")

print("="*80)

# ====================================
# 2. Best Model Declaration
# ====================================
print(f"\n🏆 BEST OVERALL MODEL: {best_model}")
print("="*80)
print(f"\nTest F1-Score: {best_f1:.4f}")
print(f"CV F1-Score: {best_cv_f1:.4f}")
print("\nThis model was selected based on:")
print("  1. Highest cross-validation F1-score (robust generalization)")
print("  2. Strong test set performance")
print("  3. Balanced precision and recall")

# ====================================
# 3. Optimization Impact Analysis
# ====================================
print("\n\n📊 HYPERPARAMETER OPTIMIZATION IMPACT")
print("="*80)

print("\nRandom Forest:")
rf_improvement = (rf_optimized_metrics['f1_score'] - rf_metrics['f1_score']) * 100
print(f"  • Baseline F1-Score:   {rf_metrics['f1_score']:.4f}")
print(f"  • Optimized F1-Score:  {rf_optimized_metrics['f1_score']:.4f}")
print(f"  • Improvement:         {rf_improvement:+.2f}%")
if rf_improvement > 0:
    print("  ✅ Hyperparameter tuning improved performance")
else:
    print("  ℹ️  Baseline was already well-tuned")

print("\nLogistic Regression:")
lr_improvement = (lr_optimized_metrics['f1_score'] - lr_metrics['f1_score']) * 100
print(f"  • Baseline F1-Score:   {lr_metrics['f1_score']:.4f}")
print(f"  • Optimized F1-Score:  {lr_optimized_metrics['f1_score']:.4f}")
print(f"  • Improvement:         {lr_improvement:+.2f}%")
if lr_improvement > 0:
    print("  ✅ Hyperparameter tuning improved performance")
else:
    print("  ℹ️  Baseline was already well-tuned")

# ====================================
# 4. Overfitting / Underfitting Analysis
# ====================================
print("\n\n🔍 GENERALIZATION ANALYSIS")
print("="*80)

# Check training performance for optimized models
rf_opt_train_pred = optimized_rf_classifier.predict(X_train_scaled)
rf_opt_train_accuracy = accuracy_score(y_train, rf_opt_train_pred)

lr_opt_train_pred = optimized_lr_classifier.predict(X_train_scaled)
lr_opt_train_accuracy = accuracy_score(y_train, lr_opt_train_pred)

print("\nOptimized Random Forest:")
print(f"  • Training Accuracy:   {rf_opt_train_accuracy:.4f}")
print(f"  • Test Accuracy:       {rf_optimized_metrics['accuracy']:.4f}")
print(f"  • Difference:          {(rf_opt_train_accuracy - rf_optimized_metrics['accuracy']):.4f}")

if rf_opt_train_accuracy - rf_optimized_metrics['accuracy'] > 0.10:
    print("  ⚠️  OVERFITTING DETECTED (train >> test)")
elif rf_optimized_metrics['accuracy'] < 0.70:
    print("  ⚠️  UNDERFITTING DETECTED (both low)")
else:
    print("  ✅ Good generalization")

print("\nOptimized Logistic Regression:")
print(f"  • Training Accuracy:   {lr_opt_train_accuracy:.4f}")
print(f"  • Test Accuracy:       {lr_optimized_metrics['accuracy']:.4f}")
print(f"  • Difference:          {(lr_opt_train_accuracy - lr_optimized_metrics['accuracy']):.4f}")

if lr_opt_train_accuracy - lr_optimized_metrics['accuracy'] > 0.10:
    print("  ⚠️  OVERFITTING DETECTED (train >> test)")
elif lr_optimized_metrics['accuracy'] < 0.70:
    print("  ⚠️  UNDERFITTING DETECTED (both low)")
else:
    print("  ✅ Good generalization")

# ====================================
# 5. Improvement Recommendations
# ====================================
print("\n\n🚀 RECOMMENDATIONS FOR FURTHER IMPROVEMENT")
print("="*80)

recommendations = [
    ("Feature Engineering", [
        "Add temporal features (day of week, time of day, seasonality)",
        "Create interaction features (e.g., urgency × distance)",
        "Include historical patient visit patterns",
        "Add hospital specialization matching with patient needs"
    ]),
    ("Advanced Models", [
        "Try Gradient Boosting (XGBoost, LightGBM, CatBoost)",
        "Ensemble multiple models using stacking or voting",
        "Neural Networks for complex non-linear patterns",
        "SHAP values for better model interpretability"
    ]),
    ("Data Quality", [
        "Validate and clean location data (lat/lon accuracy)",
        "Add real-time hospital capacity updates",
        "Include patient feedback and satisfaction scores",
        "Track actual vs predicted outcomes for continuous learning"
    ]),
    ("Production Deployment", [
        "Implement model monitoring and drift detection",
        "Set up A/B testing framework for new models",
        "Create API endpoint for real-time predictions",
        "Build feedback loop to retrain with new data"
    ])
]

for i, (category, suggestions) in enumerate(recommendations, 1):
    print(f"\n{i}. {category}:")
    for suggestion in suggestions:
        print(f"   • {suggestion}")

print("\n" + "="*80)
print("✅ COMPREHENSIVE ANALYSIS COMPLETED!")
print("="*80)
print(f"\n🏆 Deploy {best_model} for production use.")
print(f"   Model file: {BASE_DIR}/models/" + best_model.lower().replace(' ', '_').replace('(', '').replace(')', '') + ".pkl")

In [0]:
# =============================================================================
# CELL 13: PRODUCTION DEMO - LOAD & USE BEST MODEL
# Demonstrate how to load and use the best saved model in production
# =============================================================================

print("="*80)
print("PRODUCTION DEMO: USING BEST SAVED MODEL")
print("="*80)

print("\n👉 This demonstrates how to use the best model in production")
print("    without retraining. Load the .pkl files and make predictions!\n")

# ====================================
# 1. Load Best Model Info
# ====================================
print("1️⃣ Loading best model information...\n")

import json
with open(f"{BASE_DIR}/results/best_model_info.json", 'r') as f:
    best_model_info = json.load(f)

print(f"🏆 Best Model: {best_model_info['model_name']}")
print(f"   F1-Score: {best_model_info['f1_score']:.4f}")
print(f"   CV F1-Score: {best_model_info['cv_f1_score']:.4f}")
print(f"   Model file: {best_model_info['file_path']}")

# ====================================
# 2. Load Best Model and Scaler
# ====================================
print("\n2️⃣ Loading best model and scaler from disk...\n")

# Load the best model
with open(best_model_info['file_path'], 'rb') as f:
    loaded_best_model = pickle.load(f)
print(f"✓ Best model loaded: {best_model_info['model_name']}")

# Load Scaler
with open(f"{BASE_DIR}/models/scaler.pkl", 'rb') as f:
    loaded_scaler = pickle.load(f)
print("✓ StandardScaler loaded")

print("\n✅ Best model and configuration loaded successfully!\n")

# ====================================
# 3. Show Model Performance
# ====================================
print("="*80)
print("BEST MODEL PERFORMANCE METRICS")
print("="*80)
for metric, value in best_model_info['metrics'].items():
    print(f"{metric.replace('_', ' ').title():<15s}: {value:.4f}")

# ====================================
# 4. Make Predictions on Sample Data
# ====================================
print("\n" + "="*80)
print("MAKING PREDICTIONS ON SAMPLE DATA")
print("="*80)

# Select 10 random samples from test set
np.random.seed(42)
sample_indices = np.random.choice(len(X_test), size=10, replace=False)
X_sample = X_test.iloc[sample_indices]
y_sample = y_test.iloc[sample_indices]

print(f"\nSelected {len(X_sample)} random samples for prediction demo\n")

# Scale features
X_sample_scaled = loaded_scaler.transform(X_sample)

# Make predictions with best model
predictions = loaded_best_model.predict(X_sample_scaled)
probabilities = loaded_best_model.predict_proba(X_sample_scaled)

# Create results DataFrame
results = pd.DataFrame({
    'Sample': range(1, 11),
    'Actual': ['Recommended' if y == 1 else 'Not Recommended' for y in y_sample.values],
    'Predicted': ['Recommended' if y == 1 else 'Not Recommended' for y in predictions],
    'Confidence': [f"{prob[1]:.1%}" for prob in probabilities],
    'Correct': ['✅' if (predictions[i] == y_sample.values[i]) else '❌' for i in range(len(y_sample))]
})

print("-"*100)
print("PREDICTION RESULTS")
print("-"*100)
display(results)

# ====================================
# 5. Accuracy on Sample
# ====================================
sample_accuracy = accuracy_score(y_sample, predictions)

print(f"\n🎯 Sample Accuracy: {sample_accuracy:.1%}")
correct_predictions = (predictions == y_sample.values).sum()
print(f"   Correct predictions: {correct_predictions}/{len(y_sample)}")

# ====================================
# 6. Load All Available Models
# ====================================
print("\n" + "="*80)
print("ALL AVAILABLE MODELS")
print("="*80)

print("\nThe following models are available for production use:\n")
available_models = [
    "random_forest_baseline.pkl",
    "random_forest_optimized.pkl",
    "logistic_regression_baseline.pkl",
    "logistic_regression_optimized.pkl",
    "best_model.pkl (recommended)"
]

for i, model_file in enumerate(available_models, 1):
    print(f"  {i}. {model_file}")

print(f"\n📂 All models location: {BASE_DIR}/models/")

# ====================================
# 7. Production Usage Example
# ====================================
print("\n" + "="*80)
print("PRODUCTION USAGE EXAMPLE")
print("="*80)

print(f"""
# Example: Make predictions in production using the best model

import pickle
import pandas as pd
import json

# 1. Load best model info
with open('{BASE_DIR}/results/best_model_info.json', 'r') as f:
    best_model_info = json.load(f)

# 2. Load best model and scaler
with open(best_model_info['file_path'], 'rb') as f:
    model = pickle.load(f)
    
with open('{BASE_DIR}/models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print(f"Loaded: {{best_model_info['model_name']}}") 
print(f"F1-Score: {{best_model_info['f1_score']:.4f}}")

# 3. Prepare new data (must match training features)
new_data = pd.DataFrame([{{
    'availability_score': 0.75,
    'urgency_score': 1,
    'distance_proxy': 0.2,
    'waiting_time_score': 0.8,
    'doctor_quality_score': 0.85,
    'hospital_capacity_score': 0.9,
    'city_encoded': 5,
    'state_encoded': 12,
    # ... include all {len(X_train.columns)} features
}}])

# 4. Scale features (IMPORTANT: must scale before prediction)
X_scaled = scaler.transform(new_data)

# 5. Make prediction
prediction = model.predict(X_scaled)[0]
confidence = model.predict_proba(X_scaled)[0][1]

print(f"Recommendation: {{'Yes' if prediction == 1 else 'No'}}")
print(f"Confidence: {{confidence:.1%}}")

# 6. Use confidence threshold for decision making
if confidence >= 0.7:
    print("HIGH CONFIDENCE - Recommend this hospital")
elif confidence >= 0.5:
    print("MODERATE CONFIDENCE - Consider additional factors")
else:
    print("LOW CONFIDENCE - Not recommended")
""")

print("\n" + "="*80)
print("✅ PRODUCTION DEMO COMPLETED!")
print("="*80)

print(f"\n📦 All models saved at: {BASE_DIR}")
print("\n🚀 Models are ready for production deployment!")
print(f"\n🏆 Recommended model: {best_model_info['model_name']}")
print("\n👉 Simply load the .pkl files and make predictions - no retraining needed!")

PRODUCTION DEMO: USING BEST SAVED MODEL

👉 This demonstrates how to use the best model in production
    without retraining. Load the .pkl files and make predictions!

1️⃣ Loading best model information...

🏆 Best Model: Random Forest (Baseline)
   F1-Score: 1.0000
   CV F1-Score: 1.0000
   Model file: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/random_forest_baseline.pkl

2️⃣ Loading best model and scaler from disk...

✓ Best model loaded: Random Forest (Baseline)
✓ StandardScaler loaded

✅ Best model and configuration loaded successfully!

BEST MODEL PERFORMANCE METRICS
Accuracy       : 1.0000
Precision      : 1.0000
Recall         : 1.0000
F1 Score       : 1.0000
Roc Auc        : 1.0000

MAKING PREDICTIONS ON SAMPLE DATA

Selected 10 random samples for prediction demo

----------------------------------------------------------------------------------------------------
PREDICTION RESULTS
----------------------------------------------------------------------------------------

Sample,Actual,Predicted,Confidence,Correct
1,Recommended,Recommended,100.0%,✅
2,Recommended,Recommended,100.0%,✅
3,Not Recommended,Not Recommended,0.0%,✅
4,Recommended,Recommended,100.0%,✅
5,Recommended,Recommended,99.2%,✅
6,Recommended,Recommended,99.5%,✅
7,Recommended,Recommended,100.0%,✅
8,Not Recommended,Not Recommended,0.0%,✅
9,Not Recommended,Not Recommended,0.0%,✅
10,Not Recommended,Not Recommended,0.0%,✅



🎯 Sample Accuracy: 100.0%
   Correct predictions: 10/10

ALL AVAILABLE MODELS

The following models are available for production use:

  1. random_forest_baseline.pkl
  2. random_forest_optimized.pkl
  3. logistic_regression_baseline.pkl
  4. logistic_regression_optimized.pkl
  5. best_model.pkl (recommended)

📂 All models location: /Workspace/Users/tayebielouai@gmail.com/ML/AI/models/

PRODUCTION USAGE EXAMPLE

# Example: Make predictions in production using the best model

import pickle
import pandas as pd
import json

# 1. Load best model info
with open('/Workspace/Users/tayebielouai@gmail.com/ML/AI/results/best_model_info.json', 'r') as f:
    best_model_info = json.load(f)

# 2. Load best model and scaler
with open(best_model_info['file_path'], 'rb') as f:
    model = pickle.load(f)
    
with open('/Workspace/Users/tayebielouai@gmail.com/ML/AI/models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print(f"Loaded: {best_model_info['model_name']}") 
print(f"F1-Score: {best_mo